In [4]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.3 MB/s eta 0:00:00


In [5]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks/Diego/Paper Interpretability')
print(os.getcwd())
import numpy as np
import pandas as pd
from Model_utils.utils import get_segmented_data
import pickle
from Model_utils.evaluation_pipeline import train_tdah_with_best_optuna_config,evaluate_all_windows_to_single_csv

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Diego/Paper Interpretability


In [ ]:
cpath=''
with open(
    "Data/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data(path_adhd='Data/TDAH/ieee/ADHD_group',
                                       path_control='Data/TDAH/ieee/Control_group')

base_model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
}
model_name="shallowconvnet"#shallowconvnet,tgarnet
journal_working=f"Models/TDAH/Optuna/study_{model_name}_TDAH.journal"

# El study_name debe ser solo el identificador del estudio, no la ruta completa
study_identifier = f"study_{model_name}_TDAH"

results = train_tdah_with_best_optuna_config(
    model_name=model_name,
    study_name=study_identifier, # ¡Aquí estaba el problema! Ahora es solo el identificador.
    journal_file=journal_working,
    base_model_args=base_model_args,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    epochs=2,
    batch_size=16,
    seed=34,
    seed_gap=100,
    n_repeats=2,
)


DEBUG: Intentando cargar el estudio 'study_shallowconvnet_TDAH' del journal 'Models/TDAH/Optuna/study_shallowconvnet_TDAH.journal'
DEBUG: Journal file 'Models/TDAH/Optuna/study_shallowconvnet_TDAH.journal' existe. Tamaño: 60851 bytes.
DEBUG: Estudio 'study_shallowconvnet_TDAH' cargado exitosamente. Número de trials: 20
Best trial: 19
Best value: 0.8296261433753429
Best params: {'pool_size': 24, 'pool_stride': 2, 'n_filters': 48, 'kernel_length': 88, 'dropoutRate': 0.20000000000000004, 'norm_rate': 0.75, 'learning_rate': 0.001}


In [ ]:
df_predictions = results["subject_predictions_df"]
df_predictions
#df_predictions.to_csv(f'Results_{model_name}.csv',index=False)

,model,repeat,fold,model_seed,subject,n_segments,true_class,pred_majority_class,segment_accuracy,prob_class_0,prob_class_1
0,shallowconvnet,0,0,34,v107,76,0,1,0.118421,1.178906e-01,0.882109
1,shallowconvnet,0,0,34,v108,73,0,1,0.109589,1.124012e-01,0.887599
2,shallowconvnet,0,0,34,v121,62,0,1,0.000000,5.707328e-07,0.999999
3,shallowconvnet,0,0,34,v127,56,0,1,0.017857,1.785713e-02,0.982143
4,shallowconvnet,0,0,34,v129,42,0,1,0.071429,7.003161e-02,0.929968
...,...,...,...,...,...,...,...,...,...,...,...
233,shallowconvnet,1,4,10434,v307,88,0,1,0.011364,1.136160e-02,0.988638
234,shallowconvnet,1,4,10434,v34p,75,1,1,0.986667,1.332784e-02,0.986672
235,shallowconvnet,1,4,10434,v49p,64,0,1,0.015625,1.562500e-02,0.984375
236,shallowconvnet,1,4,10434,v53p,73,0,1,0.095890,9.192404e-02,0.908076


In [ ]:
df_test = evaluate_all_windows_to_single_csv(
    windows=("2.5-5", "0-7"),
    models=("eegnet", "shallowconvnet", "tgarnet"),
    subjects=None,
    seed=42,
)


EVALUANDO VENTANA: 2.5-5

Modelo: eegnet
Sujetos: [43]

Modelo: shallowconvnet
Sujetos: [43]

Modelo: tgarnet
Sujetos: [43]

EVALUANDO VENTANA: 0-7

Modelo: eegnet
Sujetos: [43]

Modelo: shallowconvnet
Sujetos: [43]

Modelo: tgarnet
Sujetos: [43]


In [ ]:
df_test
#df_test.to_csv("Results/MI_results.csv", index=False)

,window,model,subject,n_folds,seed,mean_accuracy,std_accuracy,mean_recall,std_recall,mean_precision,std_precision,mean_kappa,std_kappa,mean_auc,std_auc
0,2.5-5,eegnet,43,5,42,0.990,0.013693,0.990,0.013693,0.990476,0.013041,0.98,0.027386,0.9990,0.002236
1,2.5-5,shallowconvnet,43,5,42,0.985,0.022361,0.985,0.022361,0.985238,0.022234,0.97,0.044721,0.9980,0.004472
2,2.5-5,tgarnet,43,5,42,0.980,0.020917,0.980,0.020917,0.980476,0.020784,0.96,0.041833,0.9995,0.001118
3,0-7,eegnet,43,5,42,0.995,0.011180,0.995,0.011180,0.995238,0.010648,0.99,0.022361,1.0000,0.000000
4,0-7,shallowconvnet,43,5,42,0.995,0.011180,0.995,0.011180,0.995238,0.010648,0.99,0.022361,0.9990,0.002236
5,0-7,tgarnet,43,5,42,0.915,0.033541,0.915,0.033541,0.920868,0.029089,0.83,0.067082,0.9830,0.007583
